# FTIR Plastic Classification – Documented Research Notebook

This notebook documents the full code pipeline used in the FTIR-based plastic classification experiment. It includes preprocessing, PCA transformation, model training, validation, and scoring of unknown samples using machine learning techniques (Random Forest, SVM, Logistic Regression).

Plastics studied: **Polyethylene (PE), Polypropylene (PP), Polystyrene (PS)**


### Section 1: Data Preparation

In [ ]:
# 1.1 Filter Known Plastic Samples
import pandas as pd

def filter_plastic_types(input_file, output_file, valid_plastic_types):
    df = pd.read_csv(input_file)
    columns_to_keep = ['Numero de onda'] + [
        col for col in df.columns 
        if any(plastic_type in col for plastic_type in valid_plastic_types) 
        and not any(col.startswith('PET') for plastic_type in valid_plastic_types)
    ]
    filtered_df = df[columns_to_keep]
    filtered_df.to_csv(output_file, index=False)

In [ ]:
# 1.2 Extract and Format .dpt Files to DataFrame
import os
import re

def extract_data_to_df(routepath, name):
    lista_muestras = os.listdir(routepath)
    data_frames = []
    for filename in lista_muestras:
        if filename.endswith(".dpt"):
            clean_name = re.sub(r'\.dpt', '', filename)
            muestra = pd.read_csv(os.path.join(routepath, filename), sep=",", header=None)
            muestra = muestra.iloc[:, 1].tolist()
            df = pd.DataFrame({clean_name: muestra})
            data_frames.append(df)
    data_samples = pd.concat(data_frames, axis=1)
    blanco = pd.read_csv(os.path.join(routepath, lista_muestras[0]), sep="\t", header=None)
    numero_onda = blanco.iloc[:, 0].tolist()
    data_samples.insert(0, "Numero de onda", numero_onda)
    data_samples.to_csv(f"{name}.csv", index=False)
    return data_samples


In [ ]:
# 1.3 Remove Outliers
def remove_outliers(df):
    data = df.copy()
    outliers_list = []

    for n in range(data.shape[1]):
        try:
            if pd.to_numeric(data.iloc[0, n]) < 0.8:
                outliers_list.append(n)
        except ValueError:
            pass

    for i in range(data.shape[1]):
        for j in range(data.shape[0]):
            try:
                if pd.to_numeric(data.iloc[j, i]) > 1.1:
                    outliers_list.append(i)
            except ValueError:
                pass

    outliers_list = list(set(outliers_list))
    data.drop(data.columns[outliers_list], axis=1, inplace=True)
    return data


In [ ]:
# 1.4 Plot FTIR Spectra
import matplotlib.pyplot as plt

def plot_dataframe(df, name):
    data_columns = [col for col in df.columns if col != "Numero de onda"]
    plt.figure(figsize=(10, 6))
    for col in data_columns:
        plt.plot(df["Numero de onda"], df[col])
    plt.xlabel("Numero de onda")
    plt.ylabel("Data")
    plt.title(f"DataFrame Plot - {name}")
    plt.grid(True)
    plt.show()


### Section 2: PCA Application

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import pickle as pk

# 2.1 Apply PCA
def apply_pca(df, components, test_df):
    traspose_dataset = df.T.reset_index().iloc[1:, 1:]
    traspose_test_dataset = test_df.T.reset_index().iloc[1:, 1:]

    std_scaler = StandardScaler()
    scaled_data = std_scaler.fit_transform(traspose_dataset)
    test_scaled_data = std_scaler.transform(traspose_test_dataset)

    pca = PCA(n_components=components)
    normalized_df = pd.DataFrame(pca.fit_transform(scaled_data), columns=[f'PC{i}' for i in range(1, components + 1)])
    normalized_test_df = pd.DataFrame(pca.transform(test_scaled_data), columns=[f'PC{i}' for i in range(1, components + 1)])

    pk.dump(std_scaler, open("std_scaler.pkl", "wb"))
    pk.dump(pca, open("pca.pkl", "wb"))

    return normalized_df, normalized_test_df


In [ ]:
# 2.2 Load and Apply PCA to New Data
def load_and_apply_pca(test_df, std_scaler_path, pca_path):
    traspose_test_dataset = test_df.T.reset_index()
    sample_tags = traspose_test_dataset.iloc[1:, 0]
    traspose_test_dataset = traspose_test_dataset.iloc[1:, 1:]

    std_scaler = pk.load(open(std_scaler_path, 'rb'))
    pca = pk.load(open(pca_path, 'rb'))

    test_scaled_data = std_scaler.transform(traspose_test_dataset)
    normalized_test_df = pd.DataFrame(pca.transform(test_scaled_data), columns=[f'PC{i}' for i in range(1, pca.n_components_ + 1)])
    normalized_test_df['Sample'] = sample_tags.reset_index(drop=True)
    
    return normalized_test_df


### Section 3: Model Training

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from joblib import dump

# Load training data
df = pd.read_csv('training-set-be.csv')
X = df[[f'PC{i}' for i in range(1, 6)]]
y = df['Label']

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define and evaluate models
models = {
    'RandomForest': (RandomForestClassifier(), {'n_estimators': [50, 100, 200]}),
    'SVM': (SVC(probability=True), {'C': [1, 10, 100], 'kernel': ['linear', 'rbf']}),
    'LogisticRegression': (LogisticRegression(), {'C': [0.1, 1, 10]})
}

# Grid search and evaluation
for name, (model, params) in models.items():
    grid = GridSearchCV(model, params, cv=5, scoring='accuracy')
    grid.fit(X_train, y_train)
    print(f"Best params for {name}: {grid.best_params_}, CV Accuracy: {grid.best_score_:.4f}")

for name, (model, _) in models.items():
    model.fit(X_train, y_train)
    test_acc = accuracy_score(y_test, model.predict(X_test))
    print(f"Test Accuracy for {name}: {test_acc:.4f}")
    dump(model, f'models/{name}_model.joblib')


### Section 4: Validation and Prediction

In [ ]:
from joblib import load
from sklearn.metrics import confusion_matrix

models = {
    'RandomForest': load('models/RandomForest_model.joblib'),
    'SVM': load('models/SVM_model.joblib'),
    'LogisticRegression': load('models/LogisticRegression_model.joblib')
}

df = pd.read_csv('blancos_df.csv')
X_val = df[[f'PC{i}' for i in range(1, 6)]]
y_val = df['Label']

for name, model in models.items():
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    print(f"{name} Validation Accuracy: {acc:.4f}")
    print(f"Confusion Matrix for {name}:\n{confusion_matrix(y_val, y_pred)}")